<img align="right" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="left">Train & Evaluate YOLO Model</h1>
<h4 align="left">Written by the KSO Team</h3>

This notebook trains (or fine-tunes) a YOLO object-detection model using [Ultralytics](https://docs.ultralytics.com/) directly.
It is designed to work with any compatible [YOLO dataset](https://docs.ultralytics.com/datasets/detect/) and supports:

- **Training from scratch** using Ultralytics baseline models
- **Fine-tuning** from your own pretrained `.pt` weights

### Quick Start

1. Prepare your dataset (run the `Biigle_to_YOLO` notebook, or bring your own YOLO-format dataset)
2. Update the **Phase 1: Configuration** cell with your paths and settings
3. Run all cells top to bottom
4. Use the test-set metrics (Phase 4) to evaluate your model

---

### Available Baseline Models

| Family | Sizes (n / s / m / l / x) |
|--------|---------------------------|
| **YOLOv8** | `yolov8n.pt` &nbsp; `yolov8s.pt` &nbsp; `yolov8m.pt` &nbsp; `yolov8l.pt` &nbsp; `yolov8x.pt` |
| **YOLOv9** | `yolov9t.pt` &nbsp; `yolov9s.pt` &nbsp; `yolov9m.pt` &nbsp; `yolov9c.pt` &nbsp; `yolov9e.pt` |
| **YOLOv10** | `yolov10n.pt` &nbsp; `yolov10s.pt` &nbsp; `yolov10m.pt` &nbsp; `yolov10l.pt` &nbsp; `yolov10x.pt` |
| **YOLO11** | `yolo11n.pt` &nbsp; `yolo11s.pt` &nbsp; `yolo11m.pt` &nbsp; `yolo11l.pt` &nbsp; `yolo11x.pt` |

> To fine-tune from your own model, set `baseline_weights` to the path of your `.pt` file.

### Model Size Tips

- **Small datasets (~100-250 frames):** nano or small — larger models will overfit
- **Medium datasets (~250-750 frames):** medium for a good balance
- **Large datasets (750+ frames):** large or xlarge for best accuracy

---
## Phase 1: Configuration

Edit these values to match your setup, then run the cell. You need to set four things:
- `data_path` —> absolute path to your YOLO dataset folder (the one containing `data.yaml`)
- `EXPERIMENT_ROOT` —> absolute path to the directory where training runs will be saved
- `exp_name` —> a name for this experiment (creates a subfolder)
- `baseline_weights` —> Name of the Ultralytics model you're using (see `Available baseline models` table above), or a path to your own .pt weights

> For the full list of training parameters, see the [Ultralytics training arguments documentation](https://docs.ultralytics.com/modes/train/#arguments).

In [ ]:
from pathlib import Path
import re

# ── Paths ──
data_path = (
    Path("<path-to-your-dataset>").expanduser().resolve()
)  # Path where your dataset and data.yaml are located
EXPERIMENT_ROOT = (
    Path("<path-to-your-models-dir>").expanduser().resolve()
)  # Where training runs are saved

# ── Experiment ──
exp_name = "<your-experiment-name>"  # Just a name for this run, not a path. Change each time you make a new run.
exp_name = re.sub(
    r"[^\w\-.]", "_", exp_name.strip()
)  # Sanitise: spaces/special chars → underscores
baseline_weights = (
    "yolo11m.pt"  # Ultralytics model name OR absolute path to your own .pt weights
)

# ── Training ──
epochs = 100  # Recommended: 50-150 depending on dataset size
batch_size = 8  # Reduce if you run out of GPU memory
img_size = 640  # Standard YOLO input size

# ── Evaluation ──
conf_thres = 0.5  # Confidence threshold for test evaluation

# ── Derived (no need to edit) ──
if "/" in baseline_weights or "\\" in baseline_weights:
    baseline_weights = str(Path(baseline_weights).expanduser().resolve())
data_yaml = data_path / "data.yaml"

print(f"Dataset:   {data_path}")
print(f"Output:    {EXPERIMENT_ROOT / exp_name}")
print(f"Model:     {baseline_weights}")
print(f"Epochs:    {epochs}")

---
## Phase 2: Verify Setup

Checks that paths exist and the environment is ready.

In [ ]:
import os
import torch
from ultralytics import settings as ultra_settings

# Disable wandb to avoid login prompts / crashes
ultra_settings.update({"wandb": False})

# Compute safe worker count (half of available CPUs, minimum 1)
try:
    NUM_WORKERS = max(1, len(os.sched_getaffinity(os.getpid())) // 2)
except AttributeError:
    # macOS / systems without sched_getaffinity
    NUM_WORKERS = min(4, os.cpu_count() or 4)

# Check dataset
assert data_path.exists(), f"Dataset folder not found: {data_path}"
assert data_yaml.exists(), f"data.yaml not found: {data_yaml}"
print(f"Dataset:   {data_path}")

# Check for expected splits
for split in ("train", "valid"):
    split_dir = data_path / split
    assert split_dir.exists(), f"Required split missing: {split_dir}"
    n_images = (
        len(list((split_dir / "images").glob("*")))
        if (split_dir / "images").exists()
        else 0
    )
    print(f"  {split + ':':10s} {n_images} images")

# Check test split (optional but needed for Phase 4)
test_dir = data_path / "test"
if test_dir.exists():
    n_test = (
        len(list((test_dir / "images").glob("*")))
        if (test_dir / "images").exists()
        else 0
    )
    print(f"  {'test:':10s} {n_test} images")
else:
    print("test: NOT FOUND (Phase 4 will be skipped)")

# Check GPU
if torch.cuda.is_available():
    print(f"GPU:   {torch.cuda.get_device_name(0)}")
else:
    print("GPU:   None detected (training will be slow)")

# Create output directory
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Output dir:   {EXPERIMENT_ROOT}")
print(f"Workers:   {NUM_WORKERS}")
print()
print("Ready to train.")

---
## Phase 3: Train Model

Trains the YOLO model on your dataset. Duration depends on dataset size, model, and number of epochs.

Results are saved to `EXPERIMENT_ROOT/<exp_name>/`. If a directory with that name already exists, 
an error will be raised — change `exp_name` in Phase 1 to give each experiment a unique name.

In [ ]:
# Check for existing experiment directory
exp_dir = EXPERIMENT_ROOT / exp_name
if exp_dir.exists() and any(exp_dir.iterdir()):
    raise FileExistsError(
        f"Experiment directory already exists and is not empty: {exp_dir}\n"
        f"Change exp_name in Phase 1 to start a new experiment, "
        f"or delete the directory to reuse this name."
    )

from ultralytics import YOLO

print(f"Training {baseline_weights} for {epochs} epochs...")
print(f"Base output: {EXPERIMENT_ROOT}")
print("=" * 50, "\n")

model = YOLO(baseline_weights)
results = model.train(
    data=str(data_yaml),
    epochs=epochs,
    batch=batch_size,
    imgsz=img_size,
    workers=NUM_WORKERS,
    project=str(EXPERIMENT_ROOT),
    name=exp_name,
    plots=True,
)

train_dir = Path(results.save_dir)
best_weights = train_dir / "weights" / "best.pt"

print(f"\n{'=' * 50}")
print(f"Training complete!")
print(f"  Run directory: {train_dir}")
print(f"  Best weights:  {best_weights}")

---
## Phase 4: Test Evaluation

Evaluates the best checkpoint on the held-out **test** split.
These are your publishable, unbiased metrics. Report them in your paper or thesis.

> If your dataset has no `test/` split, this phase will be skipped.

> **Already have trained weights?** Make sure `data_path` and `conf_thres` are set in Phase 1. You can skip Phases 2–3 and run this cell directly — just set `train_dir` manually in the code cell below (uncomment and edit with your `exp_name`)

In [ ]:
# Check that test split exists
test_dir = data_path / "test"
if not test_dir.exists():
    print("No test split found — skipping test evaluation.")
    print("To enable this, add a test/ folder to your dataset and re-run.")
else:
    # If you ran Phase 3 in this session, train_dir is already set.
    # To evaluate a previous run (skipping training), uncomment and edit:
    # train_dir = EXPERIMENT_ROOT / "your_run_folder"

    best_weights = train_dir / "weights" / "best.pt"
    assert best_weights.exists(), f"Weights not found: {best_weights}"

    print(f"Run dir: {train_dir}")
    print(f"Model:   {best_weights}")
    print(f"Evaluating on test set (conf={conf_thres})...\n")

    model = YOLO(str(best_weights))
    test_results = model.val(
        data=str(data_yaml),
        split="test",
        conf=conf_thres,
        workers=NUM_WORKERS,
        project=str(train_dir),
        name="test_eval",
        exist_ok=True,
        plots=True,
    )

    print(f"\n{'=' * 50}")
    print("TEST SET RESULTS")
    print("=" * 50)
    print(f"  Precision:  {test_results.box.mp:.3f}")
    print(f"  Recall:     {test_results.box.mr:.3f}")
    print(f"  mAP@50:     {test_results.box.map50:.3f}")
    print(f"  mAP@50-95:  {test_results.box.map:.3f}")
    print("=" * 50)
    print(f"\nResults saved to: {train_dir / 'test_eval'}")

---
## ✅ Done

### Output Structure

```
models/exp_name/
    weights/
        best.pt              # Best checkpoint — use this for inference
        last.pt              # Final checkpoint
    test_eval/               # Publishable results
        confusion_matrix.png
        PR_curve.png
        P_curve.png
        R_curve.png
        F1_curve.png
    results.png              # Training curves (for monitoring only)
```
### Rerun Behavior

Each experiment must have a unique `exp_name`. If you rerun training, change `exp_name` in Phase 1 
first. To evaluate a previous run after a kernel restart, set `train_dir` manually in Phase 4 and uncomment the line.

### Which Metrics to Report?

| Location | Purpose | Publishable? |
|----------|---------|:------------:|
| `<exp_name>/results.png` | Training progress (loss curves) | No |
| `<exp_name>/test_eval/` | Unbiased test-set evaluation | **Yes** |

---
### Using Your Model for Inference

```python
from ultralytics import YOLO

model = YOLO("path/to/your/models/<exp_name>/weights/best.pt")
results = model.predict("path/to/image.jpg")
```
### Next Steps

1. **Adjust hyperparameters** in Phase 1 and re-run training
2. **Compare experiments** by checking metrics across different `exp_name` runs
3. **Navigate to the next Notebook** `Model_Inference` 